In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__results__.html
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__notebook__.ipynb
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/__output__.json
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/tfidf.pkl
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/custom.css
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/repo/README.md
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/repo/DL-24f2001460-notebook-t22026.ipynb
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/repo/models/tfidf.py
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/repo/models/scratch.py
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/repo/models/__pycache__/tfidf.cpython-312.pyc
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/repo/wandb/run-20260715_164957-sne2vidl/run-sne2vidl.wandb
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/repo/wandb/run-20260715_164957-sne2vidl/logs/debug.log
/kaggle/input/notebooks/jigyasadoharey/tfidf-model/repo

In [2]:
# import numpy as np
# !pip install -q -U bitsandbytes>=0.46.1
# import pandas as pd
# import torch
# import os
# import wandb
# from datasets import Dataset
# from transformers import (
#     AutoTokenizer,
#     AutoModelForCausalLM,
#     BitsAndBytesConfig,
#     TrainingArguments,
#     Trainer,
#     DataCollatorForSeq2Seq,
# )
# from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# try:
#     from kaggle_secrets import UserSecretsClient
#     wandb_key = UserSecretsClient().get_secret("wandb_key")
# except Exception:
#     wandb_key = os.environ.get("wandb_key")

# wandb.login(key=wandb_key)
# wandb.init(project="24f2001460-t22026", name="qwen2.5-7b-qlora-mc")

# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# MAX_LEN = 512
# OPTIONS = ["A", "B", "C", "D", "E"]

# train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
# test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# tokenizer.pad_token = tokenizer.eos_token
# tokenizer.padding_side = "right"

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     quantization_config=bnb_config,
#     device_map="auto",
# )
# model = prepare_model_for_kbit_training(model)

# lora_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
# )
# model = get_peft_model(model, lora_config)

# OPTION_TOKEN_IDS = {o: tokenizer.encode(o, add_special_tokens=False)[0] for o in OPTIONS}

# def build_prompt(row):
#     return (
#         f"Question: {row['prompt']}\n"
#         f"A. {row['A']}\n"
#         f"B. {row['B']}\n"
#         f"C. {row['C']}\n"
#         f"D. {row['D']}\n"
#         f"E. {row['E']}\n"
#         f"Answer:"
#     )

# def preprocess_train(examples):
#     input_ids_list, labels_list, attn_list = [], [], []
#     for i in range(len(examples["prompt"])):
#         row = {k: examples[k][i] for k in examples}
#         prompt = build_prompt(row)
#         full_text = prompt + " " + row["answer"]

#         prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
#         full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
#         full_ids = full_ids[:MAX_LEN]

#         labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
#         labels = labels[:MAX_LEN]

#         attn = [1] * len(full_ids)

#         input_ids_list.append(full_ids)
#         labels_list.append(labels)
#         attn_list.append(attn)

#     return {"input_ids": input_ids_list, "labels": labels_list, "attention_mask": attn_list}

# train_dataset = Dataset.from_pandas(train_df)
# train_dataset = train_dataset.map(
#     preprocess_train, batched=True, remove_columns=train_df.columns.tolist()
# )

# split = train_dataset.train_test_split(test_size=0.1, seed=42)
# train_ds, val_ds = split["train"], split["test"]

# data_collator = DataCollatorForSeq2Seq(
#     tokenizer=tokenizer, padding=True, label_pad_token_id=-100
# )

# args = TrainingArguments(
#     output_dir="/kaggle/working/mcq_qwen_output",
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     learning_rate=2e-4,
#     per_device_train_batch_size=1,
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=16,
#     num_train_epochs=3,
#     weight_decay=0.01,
#     warmup_ratio=0.05,
#     bf16=True,
#     optim="paged_adamw_8bit",
#     report_to="wandb",
#     logging_steps=10,
#     save_total_limit=1,
#     load_best_model_at_end=True,
# )

# trainer = Trainer(
#     model=model,
#     args=args,
#     train_dataset=train_ds,
#     eval_dataset=val_ds,
#     processing_class=tokenizer,
#     data_collator=data_collator,
# )

# trainer.train()

# model.eval()
# option_ids_tensor = torch.tensor([OPTION_TOKEN_IDS[o] for o in OPTIONS])

# def predict_top3(row):
#     prompt = build_prompt(row) + " "
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     with torch.no_grad():
#         logits = model(**inputs).logits[0, -1]
#     option_logits = logits[option_ids_tensor.to(model.device)]
#     probs = torch.softmax(option_logits, dim=0).float().cpu().numpy()
#     top3_idx = np.argsort(-probs)[:3]
#     return " ".join(OPTIONS[i] for i in top3_idx)

# test_ids = test_df["id"].tolist()
# predictions = [predict_top3(row) for _, row in test_df.iterrows()]

# submission = pd.DataFrame({"id": test_ids, "Prediction": predictions})
# submission.to_csv("submission.csv", index=False)

# wandb.finish()

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("new_token")


In [4]:
!git clone https://{secret_value_0}@github.com/24f2001460/dl-24f2001460-notebook-t22026.git /kaggle/working/repo1
%cd /kaggle/working/repo1

Cloning into '/kaggle/working/repo1'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 121 (delta 58), reused 80 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 96.02 KiB | 5.33 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/kaggle/working/repo1


In [5]:
# ── 3. Run inference ───────────────────────────────────────────────────────
# Adjust --model_path to wherever Kaggle mounted your saved tfidf.pkl dataset
!python -m src.tfidf_inference\
    --test_path  /kaggle/input/competitions/smart-mcq-solver-challenge/test.csv \
    --model_path  /kaggle/input/notebooks/jigyasadoharey/tfidf-model/tfidf.pkl \
    --output_path /kaggle/working/submission.csv

/kaggle/working/repo1/src/tfidf_inference.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = expanded.groupby('id').apply(rank_options).reset_index()
Saved 500 predictions to /kaggle/working/submission.csv


In [6]:
# !python -m src.scratch_inference\
#     --test_path  /kaggle/input/competitions/smart-mcq-solver-challenge/test.csv \
#     --model_path  /kaggle/input/notebooks/jigyasadoharey/scratch-model/model.pt\
#     --vocab_path /kaggle/input/notebooks/jigyasadoharey/scratch-model/word2idx.pkl\
#     --output_path /kaggle/working/submission.csv